# Alternative CPA Pathways Survey – Cross-Sectional AnalysisThis notebook loads the Qualtrics export, auto-detects relevant questions by text, and produces Q1/Q2 charts and summaries.

In [ ]:
import osimport reimport textwrapfrom collections import Counterimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy import statsOUTPUT_DIR = './outputs'REPORT_DIR = './reports'os.makedirs(OUTPUT_DIR, exist_ok=True)os.makedirs(REPORT_DIR, exist_ok=True)plt.rcParams.update({    'figure.dpi': 120,    'savefig.dpi': 300,    'font.size': 11,    'axes.titlesize': 13,    'axes.labelsize': 11,    'legend.fontsize': 10,})

In [ ]:
DATA_PATH = 'Alternative CPA Pathways Survey_December 31, 2025_09.45.csv'QUALTRICS_META_COLS = {    'StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress', 'Finished',    'RecordedDate', 'ResponseId', 'RecipientLastName', 'RecipientFirstName',    'RecipientEmail', 'ExternalReference', 'LocationLatitude', 'LocationLongitude',    'DistributionChannel', 'UserLanguage'}AWARENESS_KEYWORDS = ['aware', 'awareness', 'alternative pathway', '150', 'licensure pathway']CPA_LIKELIHOOD_KEYWORDS = ['likely', 'likelihood', 'pursue', 'cpa']GRAD_DESIRE_KEYWORDS = ['graduate', 'macc', 'master', 'less likely', 'desire']SATISFACTION_KEYWORDS = ['satisfied', 'satisfaction']STUDENT_LEVEL_KEYWORDS = ['undergraduate', 'graduate', 'program level']GRAD_TRACK_KEYWORDS = ['graduate program', 'grad program', 'currently', 'considering']AWARE_WHEN_KEYWORDS = ['when', 'became aware', 'learned', 'heard']ENROLLED_KNOWING_KEYWORDS = ['enrolled', 'knowing', 'aware before', 'before enrolling']

In [ ]:
def load_qualtrics_csv(path):    header_preview = pd.read_csv(path, nrows=2, header=None)    col_ids = header_preview.iloc[0].tolist()    question_texts = header_preview.iloc[1].tolist()    data = pd.read_csv(path, header=0, skiprows=[1], dtype=str)    data.columns = col_ids    qtext_map = {col_id: (qtext if pd.notna(qtext) else str(col_id))                 for col_id, qtext in zip(col_ids, question_texts)}    return data, qtext_mapdef clean_text_fields(df):    for col in df.columns:        if df[col].dtype == object:            df[col] = df[col].str.replace(r"^'", '', regex=True)    return dfdef drop_metadata_columns(df):    return df.drop(columns=[col for col in df.columns if col in QUALTRICS_META_COLS], errors='ignore')def keyword_score(text, keywords):    hits = sum(1 for kw in keywords if kw in text)    return hitsdef find_best_column(qtext_map, keywords, exclude_keywords=None):    candidates = []    for col_id, text in qtext_map.items():        text_lower = str(text).lower()        if exclude_keywords and any(kw in text_lower for kw in exclude_keywords):            continue        score = keyword_score(text_lower, keywords)        if score > 0:            candidates.append((col_id, text, score))    candidates.sort(key=lambda x: (x[2], len(str(x[1]))), reverse=True)    best = candidates[0] if candidates else None    return best, candidatesdef coerce_likert(series):    numeric = pd.to_numeric(series, errors='coerce')    if numeric.notna().mean() >= 0.5:        return numeric    unique_vals = [val for val in series.dropna().unique()]    return pd.Categorical(series, categories=unique_vals, ordered=True)def print_selected_variables(selected):    print('Selected Variables')    print('-----------------')    for entry in selected:        print(f"- {entry['role']}: {entry['col_id']} | {entry['question_text']}")        if entry.get('reason'):            print(f"  - Reason: {entry['reason']}")    print()def summarize_candidates(candidates, max_items=5):    summary = []    for col_id, text, score in candidates[:max_items]:        summary.append(f"{col_id}: {text} (score={score})")    return summarydef compute_cohens_d(a, b):    a = a.dropna().astype(float)    b = b.dropna().astype(float)    n1, n2 = len(a), len(b)    if n1 < 2 or n2 < 2:        return np.nan    s1, s2 = a.std(ddof=1), b.std(ddof=1)    pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))    if pooled == 0:        return np.nan    return (a.mean() - b.mean()) / pooleddef compute_ci(series, confidence=0.95):    clean = series.dropna().astype(float)    n = len(clean)    if n < 2:        return (np.nan, np.nan)    mean = clean.mean()    sem = stats.sem(clean, nan_policy='omit')    h = sem * stats.t.ppf((1 + confidence) / 2., n - 1)    return (mean - h, mean + h)def cramers_v(table):    chi2, _, _, _ = stats.chi2_contingency(table)    n = table.sum().sum()    r, c = table.shape    return np.sqrt(chi2 / (n * (min(r - 1, c - 1))))def plot_distribution(series, title, filename):    clean = series.dropna()    fig, ax = plt.subplots(figsize=(7, 4))    if pd.api.types.is_numeric_dtype(clean):        ax.hist(clean.astype(float), bins=10, color='#4C72B0', edgecolor='white')        ax.set_xlabel(series.name)        ax.set_ylabel('Count')    else:        counts = clean.value_counts().sort_index()        ax.bar(counts.index.astype(str), counts.values, color='#4C72B0')        ax.set_ylabel('Count')        ax.set_xlabel(series.name)        ax.tick_params(axis='x', rotation=30)    ax.set_title(f"{title} (n={len(clean)})")    fig.tight_layout()    fig.savefig(os.path.join(OUTPUT_DIR, filename))    plt.show()def plot_means_with_ci(df, group_col, outcome_col, title, filename):    groups = df[[group_col, outcome_col]].dropna()    means = groups.groupby(group_col)[outcome_col].mean()    cis = groups.groupby(group_col)[outcome_col].apply(lambda x: compute_ci(x))    errors = np.array([(mean - ci[0], ci[1] - mean) for mean, ci in zip(means, cis)])    fig, ax = plt.subplots(figsize=(7, 4))    ax.bar(means.index.astype(str), means.values, yerr=errors.T, capsize=6, color='#55A868')    ax.set_ylabel(outcome_col)    ax.set_title(f"{title} (n={len(groups)})")    ax.tick_params(axis='x', rotation=20)    fig.tight_layout()    fig.savefig(os.path.join(OUTPUT_DIR, filename))    plt.show()def plot_stacked_percent(df, group_col, outcome_col, title, filename):    subset = df[[group_col, outcome_col]].dropna()    table = pd.crosstab(subset[group_col], subset[outcome_col])    percent = table.div(table.sum(axis=1), axis=0) * 100    fig, ax = plt.subplots(figsize=(7, 4))    bottom = np.zeros(len(percent))    for col in percent.columns:        ax.bar(percent.index.astype(str), percent[col].values, bottom=bottom, label=str(col))        bottom += percent[col].values    ax.set_ylabel('Percent')    ax.set_title(f"{title} (n={len(subset)})")    ax.legend(title=outcome_col, bbox_to_anchor=(1.05, 1), loc='upper left')    fig.tight_layout()    fig.savefig(os.path.join(OUTPUT_DIR, filename))    plt.show()def derive_awareness_group(df, awareness_col, when_col=None):    awareness = df[awareness_col].astype(str).str.lower()    aware_flag = ~awareness.str.contains(r'(not aware|unaware|no)', regex=True, na=False)    group = np.where(aware_flag, 'Aware', 'Unaware')    if when_col:        when = df[when_col].astype(str).str.lower()        group = np.where(when.str.contains(r'before grad|before graduate|prior to grad'),                         'Aware before grad', group)        group = np.where(when.str.contains(r'before survey|before taking|prior to survey'),                         'Aware before survey', group)        group = np.where(when.str.contains(r'not aware|unaware|never'),                         'Unaware', group)    return pd.Series(group, index=df.index, name='Awareness Group')

In [ ]:
df_raw, qtext_map = load_qualtrics_csv(DATA_PATH)df_raw = clean_text_fields(df_raw)df = drop_metadata_columns(df_raw)selected_vars = []aware_best, aware_candidates = find_best_column(qtext_map, AWARENESS_KEYWORDS)if not aware_best:    raise ValueError('No awareness item found. Closest matches: ' + str(summarize_candidates(aware_candidates)))aware_col = aware_best[0]selected_vars.append({    'role': 'awareness',    'col_id': aware_best[0],    'question_text': aware_best[1],    'reason': f'Highest keyword score ({aware_best[2]})'})when_best, when_candidates = find_best_column(qtext_map, AWARE_WHEN_KEYWORDS, exclude_keywords=None)when_col = Noneif when_best and any(kw in str(when_best[1]).lower() for kw in AWARENESS_KEYWORDS):    when_col = when_best[0]    selected_vars.append({        'role': 'awareness timing',        'col_id': when_best[0],        'question_text': when_best[1],        'reason': f'Timing question with awareness terms (score {when_best[2]})'    })likelihood_best, likelihood_candidates = find_best_column(qtext_map, CPA_LIKELIHOOD_KEYWORDS)if not likelihood_best:    raise ValueError('No CPA likelihood item found. Closest matches: ' + str(summarize_candidates(likelihood_candidates)))likelihood_col = likelihood_best[0]selected_vars.append({    'role': 'CPA likelihood',    'col_id': likelihood_best[0],    'question_text': likelihood_best[1],    'reason': f'Highest keyword score ({likelihood_best[2]})'})student_best, student_candidates = find_best_column(qtext_map, STUDENT_LEVEL_KEYWORDS)student_level_col = student_best[0] if student_best else Noneif student_best:    selected_vars.append({        'role': 'student level',        'col_id': student_best[0],        'question_text': student_best[1],        'reason': f'Highest keyword score ({student_best[2]})'    })grad_track_best, grad_track_candidates = find_best_column(qtext_map, GRAD_TRACK_KEYWORDS)grad_track_col = Noneif grad_track_best and (not student_level_col or grad_track_best[2] > student_best[2]):    grad_track_col = grad_track_best[0]    selected_vars.append({        'role': 'grad track',        'col_id': grad_track_best[0],        'question_text': grad_track_best[1],        'reason': f'Highest keyword score ({grad_track_best[2]})'    })grad_desire_best, grad_desire_candidates = find_best_column(qtext_map, GRAD_DESIRE_KEYWORDS)satisfaction_best, satisfaction_candidates = find_best_column(qtext_map, SATISFACTION_KEYWORDS)grad_outcome_col = Nonegrad_outcome_label = Noneif grad_desire_best:    grad_outcome_col = grad_desire_best[0]    grad_outcome_label = 'grad_desire'    selected_vars.append({        'role': 'grad outcome (primary)',        'col_id': grad_desire_best[0],        'question_text': grad_desire_best[1],        'reason': f'Prioritized reduced desire item (score {grad_desire_best[2]})'    })if satisfaction_best:    if grad_outcome_col is None:        grad_outcome_col = satisfaction_best[0]        grad_outcome_label = 'grad_satisfaction'        selected_vars.append({            'role': 'grad outcome (primary)',            'col_id': satisfaction_best[0],            'question_text': satisfaction_best[1],            'reason': f'Used satisfaction item (score {satisfaction_best[2]})'        })    else:        selected_vars.append({            'role': 'grad outcome (secondary)',            'col_id': satisfaction_best[0],            'question_text': satisfaction_best[1],            'reason': f'Secondary satisfaction item (score {satisfaction_best[2]})'        })enrolled_best, enrolled_candidates = find_best_column(qtext_map, ENROLLED_KNOWING_KEYWORDS)enrolled_knowing_col = Noneif enrolled_best:    enrolled_knowing_col = enrolled_best[0]    selected_vars.append({        'role': 'enrolled knowing',        'col_id': enrolled_best[0],        'question_text': enrolled_best[1],        'reason': f'Keyword score ({enrolled_best[2]})'    })print_selected_variables(selected_vars)df['Awareness Group'] = derive_awareness_group(df, aware_col, when_col=when_col)df['CPA Likelihood'] = coerce_likert(df[likelihood_col])if student_level_col:    df['Student Level'] = df[student_level_col]if grad_outcome_col:    df['Grad Outcome'] = coerce_likert(df[grad_outcome_col])if satisfaction_best and grad_outcome_label == 'grad_desire':    df['Grad Satisfaction'] = coerce_likert(df[satisfaction_best[0]])

## Q1: CPA Likelihood by Awareness

In [ ]:
plot_distribution(df['CPA Likelihood'], 'Overall CPA Likelihood', 'q1_overall_cpa_likelihood.png')q1_data = df[['Awareness Group', 'CPA Likelihood']].dropna()is_numeric = pd.api.types.is_numeric_dtype(q1_data['CPA Likelihood'])if is_numeric:    plot_means_with_ci(q1_data, 'Awareness Group', 'CPA Likelihood',                       'CPA Likelihood by Awareness', 'q1_by_awareness.png')    aware_vals = q1_data[q1_data['Awareness Group'] == 'Aware']['CPA Likelihood']    unaware_vals = q1_data[q1_data['Awareness Group'] == 'Unaware']['CPA Likelihood']    if len(aware_vals) > 1 and len(unaware_vals) > 1:        t_stat, p_value = stats.ttest_ind(aware_vals, unaware_vals, equal_var=False, nan_policy='omit')        d_value = compute_cohens_d(aware_vals, unaware_vals)        print(f"Q1 t-test (aware vs unaware): t={t_stat:.3f}, p={p_value:.4f}, Cohen's d={d_value:.3f}")        print(f"Group ns: aware={aware_vals.notna().sum()}, unaware={unaware_vals.notna().sum()}")else:    plot_stacked_percent(q1_data, 'Awareness Group', 'CPA Likelihood',                         'CPA Likelihood by Awareness', 'q1_by_awareness.png')    contingency = pd.crosstab(q1_data['Awareness Group'], q1_data['CPA Likelihood'])    chi2, p_value, _, _ = stats.chi2_contingency(contingency)    v_value = cramers_v(contingency)    print(f"Q1 chi-square: chi2={chi2:.3f}, p={p_value:.4f}, Cramér's V={v_value:.3f}")    print(f"Group ns: {contingency.sum(axis=1).to_dict()}")if student_level_col:    q1_level = df[['Student Level', 'Awareness Group', 'CPA Likelihood']].dropna()    levels = q1_level['Student Level'].dropna().unique()    fig, axes = plt.subplots(1, len(levels), figsize=(7 * len(levels), 4), sharey=False)    if len(levels) == 1:        axes = [axes]    for ax, level in zip(axes, levels):        subset = q1_level[q1_level['Student Level'] == level]        if pd.api.types.is_numeric_dtype(subset['CPA Likelihood']):            means = subset.groupby('Awareness Group')['CPA Likelihood'].mean()            cis = subset.groupby('Awareness Group')['CPA Likelihood'].apply(lambda x: compute_ci(x))            errors = np.array([(mean - ci[0], ci[1] - mean) for mean, ci in zip(means, cis)])            ax.bar(means.index.astype(str), means.values, yerr=errors.T, capsize=5, color='#C44E52')            ax.set_ylabel('CPA Likelihood')        else:            table = pd.crosstab(subset['Awareness Group'], subset['CPA Likelihood'])            percent = table.div(table.sum(axis=1), axis=0) * 100            bottom = np.zeros(len(percent))            for col in percent.columns:                ax.bar(percent.index.astype(str), percent[col].values, bottom=bottom, label=str(col))                bottom += percent[col].values            ax.legend(title='CPA Likelihood', fontsize=8)            ax.set_ylabel('Percent')        ax.set_title(f"{level} (n={len(subset)})")        ax.tick_params(axis='x', rotation=20)    fig.suptitle('CPA Likelihood by Awareness and Student Level')    fig.tight_layout(rect=[0, 0, 1, 0.95])    fig.savefig(os.path.join(OUTPUT_DIR, 'q1_by_awareness_student_level.png'))    plt.show()print('\nQ1 Interpretation:')print('This cross-sectional analysis describes how awareness of the alternative CPA pathway relates to self-reported likelihood of pursuing the CPA at the time of the survey. Differences reflect group distributions only, not causal change over time.')

## Q2: Graduate Track Outcomes by Awareness

In [ ]:
if grad_outcome_col is None:    raise ValueError('No graduate outcome item detected. Closest matches: ' + str(summarize_candidates(grad_desire_candidates + satisfaction_candidates)))grad_subset_reason = Noneif student_level_col:    grad_subset = df[df['Student Level'].astype(str).str.contains('grad', case=False, na=False)].copy()    grad_subset_reason = f'Using student level column {student_level_col} to define grad-track.'elif grad_track_col:    grad_subset = df[df[grad_track_col].astype(str).str.contains('grad', case=False, na=False)].copy()    grad_subset_reason = f'Using grad-track column {grad_track_col} to define grad-track.'else:    grad_subset = df.copy()    grad_subset_reason = 'No grad-track indicator found; using full sample.'print(f"Grad-track rule: {grad_subset_reason}")grad_data = grad_subset[['Awareness Group', 'Grad Outcome']].dropna()is_grad_numeric = pd.api.types.is_numeric_dtype(grad_data['Grad Outcome'])if is_grad_numeric:    plot_means_with_ci(grad_data, 'Awareness Group', 'Grad Outcome',                       'Grad Outcome by Awareness', 'q2_grad_outcome_by_awareness.png')    aware_vals = grad_data[grad_data['Awareness Group'] == 'Aware']['Grad Outcome']    unaware_vals = grad_data[grad_data['Awareness Group'] == 'Unaware']['Grad Outcome']    if len(aware_vals) > 1 and len(unaware_vals) > 1:        t_stat, p_value = stats.ttest_ind(aware_vals, unaware_vals, equal_var=False, nan_policy='omit')        d_value = compute_cohens_d(aware_vals, unaware_vals)        print(f"Q2 t-test (aware vs unaware): t={t_stat:.3f}, p={p_value:.4f}, Cohen's d={d_value:.3f}")        print(f"Group ns: aware={aware_vals.notna().sum()}, unaware={unaware_vals.notna().sum()}")else:    plot_stacked_percent(grad_data, 'Awareness Group', 'Grad Outcome',                         'Grad Outcome by Awareness', 'q2_grad_outcome_by_awareness.png')    contingency = pd.crosstab(grad_data['Awareness Group'], grad_data['Grad Outcome'])    chi2, p_value, _, _ = stats.chi2_contingency(contingency)    v_value = cramers_v(contingency)    print(f"Q2 chi-square: chi2={chi2:.3f}, p={p_value:.4f}, Cramér's V={v_value:.3f}")    print(f"Group ns: {contingency.sum(axis=1).to_dict()}")if enrolled_knowing_col:    enrolled_subset = grad_subset[[enrolled_knowing_col, 'Grad Outcome']].dropna()    if not enrolled_subset.empty:        if pd.api.types.is_numeric_dtype(enrolled_subset['Grad Outcome']):            plot_means_with_ci(enrolled_subset.rename(columns={enrolled_knowing_col: 'Enrolled Knowing'}),                               'Enrolled Knowing', 'Grad Outcome',                               'Grad Outcome by Awareness at Enrollment',                               'q2_grad_outcome_by_enrolled_knowing.png')        else:            plot_stacked_percent(enrolled_subset.rename(columns={enrolled_knowing_col: 'Enrolled Knowing'}),                                 'Enrolled Knowing', 'Grad Outcome',                                 'Grad Outcome by Awareness at Enrollment',                                 'q2_grad_outcome_by_enrolled_knowing.png')if satisfaction_best and grad_outcome_label == 'grad_desire':    sat_data = grad_subset[['Awareness Group', 'Grad Satisfaction']].dropna()    if not sat_data.empty:        if pd.api.types.is_numeric_dtype(sat_data['Grad Satisfaction']):            plot_means_with_ci(sat_data, 'Awareness Group', 'Grad Satisfaction',                               'Grad Satisfaction by Awareness', 'q2_grad_satisfaction_by_awareness.png')        else:            plot_stacked_percent(sat_data, 'Awareness Group', 'Grad Satisfaction',                                 'Grad Satisfaction by Awareness', 'q2_grad_satisfaction_by_awareness.png')print('\nQ2 Interpretation:')print('Within the grad-track subset, this cross-sectional comparison summarizes how awareness of the alternative pathway relates to self-reported graduate outcomes at the survey timepoint. Results do not imply causal change.')

## Save Markdown Summary

In [ ]:
summary_lines = [    '# Survey Summary',    '',    '## Selected Variables',    '']for entry in selected_vars:    summary_lines.append(f"- **{entry['role']}**: `{entry['col_id']}` – {entry['question_text']}")summary_lines.extend([    '',    '## Charts',    '',    '- ![Q1 Overall CPA Likelihood](../outputs/q1_overall_cpa_likelihood.png)',    '- ![Q1 CPA Likelihood by Awareness](../outputs/q1_by_awareness.png)',    '- ![Q1 CPA Likelihood by Awareness and Student Level](../outputs/q1_by_awareness_student_level.png)',    '- ![Q2 Grad Outcome by Awareness](../outputs/q2_grad_outcome_by_awareness.png)'])if enrolled_knowing_col:    summary_lines.append('- ![Q2 Grad Outcome by Awareness at Enrollment](../outputs/q2_grad_outcome_by_enrolled_knowing.png)')if satisfaction_best and grad_outcome_label == 'grad_desire':    summary_lines.append('- ![Q2 Grad Satisfaction by Awareness](../outputs/q2_grad_satisfaction_by_awareness.png)')summary_lines.extend([    '',    '## Findings',    '',    '- Q1 summarizes how awareness relates to CPA likelihood distributions at the time of the survey.',    '- Q2 summarizes awareness differences in graduate outcomes among grad-track respondents.',    '',    '## Limitations',    '',    '- Cross-sectional survey data; no longitudinal or causal inference.',    '- Self-reported change items are treated as survey responses, not observed change.'])report_path = os.path.join(REPORT_DIR, 'summary.md')with open(report_path, 'w') as f:    f.write('\n'.join(summary_lines))print(f"Report saved to {report_path}")